In [0]:
%sql
DESCRIBE categorias_raw

In [0]:
%sql
-- ¿Cuántas categorías tenemos?
SELECT COUNT(*) AS total_categorias
FROM iniciacion_deportiva.bronze.categorias_raw;

In [0]:
%sql
-- Ver estructura real — path_from_root es un array JSON con la jerarquía
SELECT categoria_id, nombre, path_from_root
FROM iniciacion_deportiva.bronze.categorias_raw
LIMIT 5;

In [0]:
%sql
-- Expandir path_from_root en columnas legibles
-- Así va a quedar en silver.categorias
WITH path_expandido AS (
    SELECT
        categoria_id,
        nombre,
        FROM_JSON(path_from_root, 'ARRAY<STRUCT<id:STRING, name:STRING>>') AS path
    FROM iniciacion_deportiva.bronze.categorias_raw
)
SELECT
    categoria_id,
    nombre,
    GET(path, 0).name AS nivel_1,
    GET(path, 1).name AS nivel_2,
    GET(path, 2).name AS nivel_3,
    GET(path, 3).name AS nivel_4,
    GET(path, 4).name AS nivel_5,
    ARRAY_JOIN(TRANSFORM(path, x -> x.name), ' > ') AS path_completo
FROM path_expandido
LIMIT 10;

In [0]:
%sql
-- ¿Cuáles son los nivel_1 (categorías raíz)?
WITH path_expandido AS (
    SELECT
        FROM_JSON(path_from_root, 'ARRAY<STRUCT<id:STRING, name:STRING>>') AS path
    FROM iniciacion_deportiva.bronze.categorias_raw
)
SELECT
    GET(path, 0).name                                  AS nivel_1,
    COUNT(*)                                           AS cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM path_expandido
GROUP BY 1
ORDER BY cantidad DESC;

In [0]:
%sql
-- ¿Cuáles son los nivel_2 más frecuentes?
WITH path_expandido AS (
    SELECT
        FROM_JSON(path_from_root, 'ARRAY<STRUCT<id:STRING, name:STRING>>') AS path
    FROM iniciacion_deportiva.bronze.categorias_raw
)
SELECT
    GET(path, 0).name AS nivel_1,
    GET(path, 1).name AS nivel_2,
    COUNT(*)          AS categorias
FROM path_expandido
GROUP BY 1, 2
ORDER BY categorias DESC
LIMIT 15;

In [0]:
%sql
-- Cruzar con ventas — ¿cuántas ventas tiene cada categoría? (cantidad)
WITH path_expandido AS (
    SELECT
        categoria_id,
        FROM_JSON(path_from_root, 'ARRAY<STRUCT<id:STRING, name:STRING>>') AS path
    FROM iniciacion_deportiva.bronze.categorias_raw
)
SELECT
    GET(path, 1).name                                                    AS nivel_2,
    COUNT(v.id_venta)                                                    AS ventas,
    ROUND(COUNT(v.id_venta) * 100.0 / SUM(COUNT(v.id_venta)) OVER(), 2) AS porcentaje
FROM path_expandido c
LEFT JOIN iniciacion_deportiva.bronze.ventas_raw v
    ON c.categoria_id = v.categoria_id
   AND v.estado = 'paid'
GROUP BY 1
ORDER BY ventas DESC;

In [0]:
%sql
-- Cruzar con ventas — ¿cuántas ventas tiene cada categoría? (Revenue)
WITH path_expandido AS (
    SELECT
        categoria_id,
        FROM_JSON(path_from_root, 'ARRAY<STRUCT<id:STRING, name:STRING>>') AS path
    FROM iniciacion_deportiva.bronze.categorias_raw
)
SELECT
    GET(path, 1).name                                                    AS nivel_2,
    SUM(precio_unitario::DOUBLE * cantidad::DOUBLE)                      AS ventas,
    ROUND(COUNT(v.id_venta) * 100.0 / SUM(COUNT(v.id_venta)) OVER(), 2) AS porcentaje
FROM path_expandido c
LEFT JOIN iniciacion_deportiva.bronze.ventas_raw v
    ON c.categoria_id = v.categoria_id
   AND v.estado = 'paid'
GROUP BY 1
ORDER BY ventas DESC;

In [0]:
%sql
SELECT MAX(ingested_at) FROM iniciacion_deportiva.bronze.ventas_raw
